# Titanic — разведочный анализ

Что смотрим:

1. что вообще в данных и где пропуски
2. как выживаемость связана с полом, классом, возрастом и ценой билета
3. что дают признаки из `src/features.py`
4. быстрый baseline, чтобы было с чем сравнивать

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# чтобы работали импорты из src/ независимо от того, откуда запущен ноутбук
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (10, 4)

train = pd.read_csv(ROOT / "data" / "raw" / "train.csv")
test = pd.read_csv(ROOT / "data" / "raw" / "test.csv")
print(train.shape, test.shape)
train.head()

## 1. Что в данных

In [ ]:
train.info()

In [ ]:
train.describe(include="all").T

### Пропуски

Age и Cabin — основные дыры. Cabin пропущен настолько часто, что сам факт
наличия каюты становится отдельным признаком (`HasCabin` в `src/features.py`).

In [ ]:
missing = pd.DataFrame(
    {
        "train_%": train.isna().mean().mul(100).round(1),
        "test_%": test.isna().mean().mul(100).round(1),
    }
).query("`train_%` > 0 or `test_%` > 0")
missing.sort_values("train_%", ascending=False)

## 2. Целевая переменная

In [ ]:
target = "Survived"
print(train[target].value_counts())
print()
print(train[target].value_counts(normalize=True).round(3))

Классы несбалансированы умеренно (примерно 62 / 38). Отсюда два следствия:
фолды делаем стратифицированными, а accuracy дополняем roc_auc — модель,
предсказывающая всем "не выжил", даст 0.62 accuracy и 0.5 roc_auc.

## 3. Выживаемость по группам

In [ ]:
for column in ["Sex", "Pclass", "Embarked", "SibSp", "Parch"]:
    rate = train.groupby(column)[target].agg(["mean", "count"]).round(3)
    print(f"--- {column} ---")
    print(rate)
    print()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, column in zip(axes, ["Sex", "Pclass", "Embarked"]):
    train.groupby(column)[target].mean().plot(kind="bar", ax=ax, rot=0)
    ax.set_title(f"Доля выживших по {column}")
    ax.set_ylabel("")
    ax.axhline(train[target].mean(), color="gray", linestyle="--", linewidth=1)
plt.tight_layout()
plt.show()

Пунктиром — средняя выживаемость по всей выборке. Пол разделяет данные сильнее
всего: это первое, на что смотрит любая модель.

In [ ]:
pd.crosstab(train["Sex"], train["Pclass"], values=train[target], aggfunc="mean").round(3)

## 4. Возраст и цена билета

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for label, group in train.groupby(target):
    axes[0].hist(group["Age"].dropna(), bins=25, alpha=0.6, label=f"Survived={label}")
axes[0].set_title("Возраст")
axes[0].legend()

for label, group in train.groupby(target):
    axes[1].hist(np.log1p(group["Fare"]), bins=25, alpha=0.6, label=f"Survived={label}")
axes[1].set_title("log(1 + Fare)")
axes[1].legend()

plt.tight_layout()
plt.show()

Цена билета сильно скошена вправо — поэтому в признаках она берётся под
логарифмом (`FareLog`). У детей до ~12 лет заметен перевес выживших: это то,
что модель вытащит через `Title=Master` и возраст.

In [ ]:
train.assign(AgeGroup=pd.cut(train["Age"], [0, 12, 18, 30, 45, 60, 100])) \
     .groupby("AgeGroup", observed=True)[target].agg(["mean", "count"]).round(3)

## 5. Признаки из src/features.py

In [ ]:
from src.features import add_features

featured = add_features(train.drop(columns=["PassengerId"]))
print("колонки после add_features:")
print(sorted(featured.columns))
featured.head()

In [ ]:
for column in ["Title", "FamilySize", "IsAlone", "HasCabin", "Deck"]:
    rate = featured.groupby(column)[target].agg(["mean", "count"]).round(3)
    print(f"--- {column} ---")
    print(rate.sort_values("mean", ascending=False))
    print()

На что смотреть: `Title` отделяет `Master` (мальчики) от `Mr` — этого не даёт
ни пол, ни возраст по отдельности. `FamilySize` работает немонотонно: одиночки
и очень большие семьи выживали хуже семей из 2-4 человек, поэтому линейной
модели этот признак даётся хуже, чем дереву.

In [ ]:
numeric = featured.select_dtypes("number")
corr = numeric.corr()[target].drop(target).sort_values(ascending=False)
corr.round(3)

Корреляция считается линейная, так что низкое значение не значит «признак
бесполезен» — `FamilySize` тому пример. Но крайние строки таблицы обычно
подсказывают, что модель будет считать важным.

## 6. Baseline

In [ ]:
import yaml

from src.training import train as run_training

with open(ROOT / "config.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg["model"] = {"name": "histgb", "params": {}}
cfg["output_dir"] = str(ROOT / "outputs")
cfg["data"]["train_path"] = str(ROOT / "data" / "raw" / "train.csv")
cfg["data"]["test_path"] = str(ROOT / "data" / "raw" / "test.csv")

metrics = run_training(cfg)
metrics

Дальше эксперименты удобнее гонять из терминала — там же они и логируются:

```bash
python main.py train --set model.name=lgbm
python main.py train --set model.name=logreg --set features.encoding=onehot --set features.scale=true
```

Результаты сравнений записывайте в `RESULTS.md`.

## Выводы

Заполните по итогам своего анализа — например:

- пол и класс дают основную часть сигнала;
- `Title` и `FamilySize` добавляют то, чего нет в исходных колонках;
- пропуски в `Age` не случайны, поэтому `AgeIsNull` имеет смысл оставить;
- что попробовать дальше: ...